In [21]:
import os
import json
import hashlib
import logging
import pandas as pd
from pathlib import Path
from datetime import datetime

In [ ]:
# 5

class PipelineError(Exception):
    """Base exception for the data pipeline."""
    pass

class SchemaError(PipelineError):
    """Raised when required columns are missing."""
    pass

class DataValidationError(PipelineError):
    """Raised when data validation fails."""
    pass

class ConfigurationError(PipelineError):
    """Raised when configuration is invalid."""
    pass

class FileProcessingError(PipelineError):
    """Raised when a file cannot be processed."""
    pass

In [ ]:
# 3, 5, 10

BASE_DIR = Path.cwd()
CONFIG_PATH = BASE_DIR / "config.json"
def load_config():
    try:
        with open(CONFIG_PATH, "r") as file:
            config = json.load(file)
        required_keys = [
            "data_directory", "output_directory", "log_directory", "manifest_file", "files"
        ]
        for key in required_keys:
            if key not in config:
                raise ConfigurationError(
                    f"Missing configuration key: {key}"
                )
        return config
    except FileNotFoundError:
        raise ConfigurationError("config.json not found")
    except json.JSONDecodeError:
        raise ConfigurationError("Invalid JSON configuration")
config = load_config()
DATA_DIR = BASE_DIR / config["data_directory"]
OUTPUT_DIR = BASE_DIR / config["output_directory"]
LOG_DIR = BASE_DIR / config["log_directory"]
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)
print("Configuration loaded successfully!")

Configuration loaded successfully!


In [ ]:
# 6

LOG_FILE = LOG_DIR / "pipeline.log"
logging.basicConfig(
    filename=str(LOG_FILE), level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", force=True
)
logger = logging.getLogger("OlistPipeline")
logger.info("Pipeline started")
print("Logging configured successfully!")

Logging configured successfully!


In [ ]:
# 2, 6

def read_all_csv_files():
    datasets = {}
    for file_path in DATA_DIR.glob("*.csv"):
        try:
            df = pd.read_csv(file_path)
            datasets[file_path.name] = df
            logger.info(
                f"Successfully read {file_path.name}"
            )
            print(
                f"{file_path.name}: {df.shape[0]} rows, "
                f"{df.shape[1]} columns"
            )
        except Exception as error:
            logger.error(
                f"Failed to read {file_path.name}: {error}"
            )
            raise FileProcessingError(
                f"Unable to read {file_path.name}: {error}"
            ) from error
    if not datasets:
        raise FileProcessingError(
            "No CSV files found in the data directory"
        )
    return datasets
datasets = read_all_csv_files()

olist_customers_dataset.csv: 99441 rows, 5 columns
olist_geolocation_dataset.csv: 1000163 rows, 5 columns
olist_orders_dataset.csv: 99441 rows, 8 columns
olist_orders_semicolon.csv: 99441 rows, 1 columns
olist_order_items_dataset.csv: 112650 rows, 7 columns
olist_order_payments_dataset.csv: 103886 rows, 5 columns
olist_order_reviews_dataset.csv: 99224 rows, 7 columns
olist_products_dataset.csv: 32951 rows, 9 columns
olist_sellers_dataset.csv: 3095 rows, 4 columns
product_category_name_translation.csv: 71 rows, 2 columns


In [35]:
# 3, 10

def apply_column_mapping(df, file_config):
    mapping = file_config.get("column_mapping", {})
    df = df.rename(columns=mapping)
    return df

In [ ]:
# 4, 5, 10

def validate_dataframe(df, filename, file_config):
    errors = []
    valid_rows = []
    df = apply_column_mapping(df.copy(), file_config)
    required_columns = file_config.get(
        "required_columns", []
    )
    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]
    if missing_columns:
        raise SchemaError(
            f"{filename}: Missing columns: {missing_columns}"
        )
    numeric_columns = file_config.get(
        "numeric_columns", []
    )
    date_columns = file_config.get(
        "date_columns", []
    )
    rules = file_config.get("rules", {})
    for index, row in df.iterrows():
        row_errors = []
        for col in numeric_columns:
            value = pd.to_numeric(
                pd.Series([row[col]]), errors="coerce"
            ).iloc[0]
            if pd.isna(value):
                row_errors.append(
                    f"{col} must be numeric"
                )
        for col in date_columns:
            value = row[col]
            if pd.notna(value):
                parsed_date = pd.to_datetime(
                    value, errors="coerce"
                )
                if pd.isna(parsed_date):
                    row_errors.append(
                        f"{col} has invalid date"
                    )
        for col in rules.get("positive", []):
            value = pd.to_numeric(
                pd.Series([row[col]]), errors="coerce"
            ).iloc[0]
            if pd.isna(value) or value <= 0:
                row_errors.append(
                    f"{col} must be greater than zero"
                )
        for col, allowed_values in rules.get(
            "allowed_values", {}
        ).items():
            if pd.notna(row[col]):
                if row[col] not in allowed_values:
                    row_errors.append(
                        f"Invalid {col}: {row[col]}"
                    )
        for col, limits in rules.get(
            "range", {}
        ).items():
            value = pd.to_numeric(
                pd.Series([row[col]]),
                errors="coerce"
            ).iloc[0]
            minimum, maximum = limits
            if (
                pd.isna(value)
                or value < minimum
                or value > maximum
            ):
                row_errors.append(
                    f"{col} must be between "
                    f"{minimum} and {maximum}"
                )
        if row_errors:
            errors.append({
                "file_name": filename,
                "row_number": index + 2,
                "reason": "; ".join(row_errors),
                "raw_record": row.to_json()
            })
        else:
            valid_rows.append(row)
    valid_df = pd.DataFrame(
        valid_rows,
        columns=df.columns
    )
    return valid_df, errors

In [ ]:
# 4, 6

def save_bad_records(all_errors):
    if not all_errors:
        print("No bad records found.")
        logger.info("No bad records found")
        return
    bad_df = pd.DataFrame(all_errors)
    output_file = OUTPUT_DIR / "bad_records.csv"
    bad_df.to_csv(
        output_file,
        index=False
    )
    logger.warning(
        f"{len(bad_df)} invalid records saved"
    )
    print(
        f"Bad records saved: {output_file}"
    )

In [ ]:
# 7

MANIFEST_PATH = BASE_DIR / config["manifest_file"]
def load_manifest():
    if MANIFEST_PATH.exists():
        with open(MANIFEST_PATH, "r") as file:
            return json.load(file)
    return {}
def save_manifest(manifest):
    with open(MANIFEST_PATH, "w") as file:
        json.dump(
            manifest, file, indent=4
        )
def calculate_file_hash(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as file:
        for chunk in iter(
            lambda: file.read(4096), b""
        ):
            sha256.update(chunk)
    return sha256.hexdigest()
def is_already_processed(
    filename, file_hash, manifest
):
    return (
        filename in manifest
        and manifest[filename]["hash"] == file_hash
        and manifest[filename]["status"] == "success"
    )

In [ ]:
# 5, 6, 7, 10

def run_pipeline():
    manifest = load_manifest()
    all_errors = []
    summary = []
    for filename, file_config in config["files"].items():
        file_path = DATA_DIR / filename
        if not file_path.exists():
            logger.warning(
                f"File not found: {filename}"
            )
            print(f"Skipping missing file: {filename}")
            continue
        try:
            file_hash = calculate_file_hash(file_path)
            if is_already_processed(
                filename, file_hash, manifest
            ):
                print(
                    f"Skipping unchanged file: {filename}"
                )
                continue
            logger.info(
                f"Processing {filename}"
            )
            df = pd.read_csv(file_path)
            valid_df, errors = validate_dataframe(
                df, filename, file_config
            )
            all_errors.extend(errors)
            output_file = (
                OUTPUT_DIR /
                f"validated_{filename}"
            )
            valid_df.to_csv(
                output_file,
                index=False
            )
            manifest[filename] = {
                "hash": file_hash,
                "status": "success",
                "processed_at": datetime.now().isoformat(),
                "total_rows": len(df),
                "valid_rows": len(valid_df),
                "invalid_rows": len(errors)
            }
            summary.append({
                "file": filename,
                "total_rows": len(df),
                "valid_rows": len(valid_df),
                "invalid_rows": len(errors)
            })
            logger.info(
                f"Completed {filename}"
            )
        except PipelineError as error:
            logger.error(
                f"Processing failed for {filename}: {error}"
            )
            manifest[filename] = {
                "hash": file_hash,
                "status": "failed",
                "error": str(error),
                "processed_at": datetime.now().isoformat()
            }
            print(
                f"Failed: {filename} - {error}"
            )
        except Exception as error:
            logger.exception(
                f"Unexpected error in {filename}"
            )
            manifest[filename] = {
                "hash": file_hash,
                "status": "failed",
                "error": str(error),
                "processed_at": datetime.now().isoformat()
            }
            print(
                f"Unexpected failure: {filename} - {error}"
            )
        finally:
            save_manifest(manifest)
    save_bad_records(all_errors)
    summary_df = pd.DataFrame(summary)
    if not summary_df.empty:
        summary_df.to_csv(
            OUTPUT_DIR / "processing_summary.csv",
            index=False
        )
        display(summary_df)
    logger.info("Pipeline execution completed")
    print("Pipeline execution completed!")

In [ ]:
# 5

try:
    run_pipeline()
except PipelineError as error:
    logger.exception(
        f"Pipeline stopped: {error}"
    )
    print(f"Pipeline error: {error}")
except Exception as error:
    logger.exception(
        f"Unexpected pipeline failure: {error}"
    )
    print(f"Unexpected error: {error}")

Skipping unchanged file: olist_customers_dataset.csv
Skipping unchanged file: olist_geolocation_dataset.csv
Skipping unchanged file: olist_order_items_dataset.csv
Skipping unchanged file: olist_order_payments_dataset.csv
Skipping unchanged file: olist_order_reviews_dataset.csv
Skipping unchanged file: olist_orders_dataset.csv
Skipping unchanged file: olist_products_dataset.csv
Skipping unchanged file: olist_sellers_dataset.csv
Skipping unchanged file: product_category_name_translation.csv
No bad records found.
Pipeline execution completed!


In [ ]:
print("Generated output files:\n")
for file in OUTPUT_DIR.iterdir():
    print(file.name)
print("\nLog file:", LOG_FILE)
print("Manifest file:", MANIFEST_PATH)

Generated output files:

bad_records.csv
processing_summary.csv
validated_olist_customers_dataset.csv
validated_olist_geolocation_dataset.csv
validated_olist_orders_dataset.csv
validated_olist_order_items_dataset.csv
validated_olist_order_payments_dataset.csv
validated_olist_order_reviews_dataset.csv
validated_olist_products_dataset.csv
validated_olist_sellers_dataset.csv
validated_product_category_name_translation.csv

Log file: c:\Users\Viveka Shree\Desktop\Vivekha\Python Training\olist_data_pipeline\logs\pipeline.log
Manifest file: c:\Users\Viveka Shree\Desktop\Vivekha\Python Training\olist_data_pipeline\manifest.json


In [ ]:
# 8

import unittest
import pandas as pd
from pathlib import Path
class TestOlistPipeline(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        """
        Load actual Olist datasets once for all tests.
        """
        cls.customers_file = DATA_DIR / "olist_customers_dataset.csv"
        cls.order_items_file = DATA_DIR / "olist_order_items_dataset.csv"
        if not cls.customers_file.exists():
            raise FileNotFoundError(
                "Olist customers CSV not found in data folder"
            )
        if not cls.order_items_file.exists():
            raise FileNotFoundError(
                "Olist order items CSV not found in data folder"
            )
        cls.customers_df = pd.read_csv(cls.customers_file)
        cls.order_items_df = pd.read_csv(cls.order_items_file)
    # TEST 1: Read all Olist CSV files
    def test_read_all_olist_csv_files(self):
        datasets = read_all_csv_files()
        self.assertGreater(len(datasets), 0)
        expected_files = set(config["files"].keys())
        self.assertTrue(
            expected_files.issubset(set(datasets.keys()))
        )
    # TEST 2: Column mapping
    def test_column_mapping(self):
        df = self.customers_df.copy()
        file_config = {
            "column_mapping": {
                "customer_city": "customer_location"
            }
        }
        result = apply_column_mapping(
            df,file_config
        )
        self.assertIn(
            "customer_location",
            result.columns
        )
        self.assertNotIn(
            "customer_city",
            result.columns
        )
    # TEST 3: Missing required column validation
    def test_missing_required_column(self):
        df = self.customers_df.copy()
        file_config = {
            "required_columns": [
                "customer_id",
                "customer_city",
                "missing_column"
            ]
        }
        with self.assertRaises(SchemaError):
            validate_dataframe(
                df,
                "olist_customers_dataset.csv",
                file_config
            )
    # TEST 4: Numeric validation
    def test_numeric_validation(self):
        # Read an actual Olist order item record
        df = self.order_items_df[
            ["price"]
        ].dropna().head(1).copy()
        # Convert column to object so it can hold text
        df["price"] = df["price"].astype("object")
        # Introduce invalid data in memory for testing
        df.loc[df.index[0], "price"] = "invalid_price"
        file_config = {
            "required_columns": ["price"],
            "numeric_columns": ["price"]
        }
        valid, errors = validate_dataframe(
            df,
            "olist_order_items_dataset.csv",
            file_config
        )
        self.assertEqual(len(valid), 0)
        self.assertEqual(len(errors), 1)
    # TEST 5: Positive value validation
    def test_positive_value_validation(self):
        # Use an actual Olist order item record
        df = self.order_items_df[
            ["price"]
        ].dropna().head(1).copy()
        # Introduce an invalid negative price in memory
        df.loc[df.index[0], "price"] = -1
        file_config = {
            "required_columns": ["price"],
            "numeric_columns": ["price"],
            "rules": {
                "positive": ["price"]
            }
        }
        valid, errors = validate_dataframe(
            df,
            "olist_order_items_dataset.csv",
            file_config
        )
        self.assertEqual(len(valid), 0)
        self.assertEqual(len(errors), 1)
    # TEST 6: File hash calculation
    def test_file_hash(self):
        file_hash = calculate_file_hash(
            self.customers_file
        )
        self.assertIsInstance(file_hash, str)
        self.assertEqual(len(file_hash), 64)
        # Verify that repeated hashing gives the same result
        second_hash = calculate_file_hash(
            self.customers_file
        )
        self.assertEqual(file_hash, second_hash)
    # TEST 7: Manifest-based idempotency
    def test_manifest_skip(self):
        filename = self.customers_file.name
        file_hash = calculate_file_hash(
            self.customers_file
        )
        manifest = {
            filename: {
                "hash": file_hash,
                "status": "success"
            }
        }
        result = is_already_processed(
            filename, file_hash, manifest
        )
        self.assertTrue(result)
# Execute all unit tests
suite = unittest.TestLoader().loadTestsFromTestCase(
    TestOlistPipeline
)
result = unittest.TextTestRunner(
    verbosity=2
).run(suite)
print("\n========== TEST SUMMARY ==========")
print("Total tests:", result.testsRun)
print("Passed:", result.testsRun - len(result.failures) - len(result.errors))
print("Failed:", len(result.failures))
print("Errors:", len(result.errors))
if result.wasSuccessful():
    print("\nAll unit tests passed successfully!")
else:
    print("\nSome unit tests failed. Check the errors above.")

test_column_mapping (__main__.TestOlistPipeline.test_column_mapping) ... ok
test_file_hash (__main__.TestOlistPipeline.test_file_hash) ... ok
test_manifest_skip (__main__.TestOlistPipeline.test_manifest_skip) ... ok
test_missing_required_column (__main__.TestOlistPipeline.test_missing_required_column) ... ok
test_numeric_validation (__main__.TestOlistPipeline.test_numeric_validation) ... ok
test_positive_value_validation (__main__.TestOlistPipeline.test_positive_value_validation) ... ok
test_read_all_olist_csv_files (__main__.TestOlistPipeline.test_read_all_olist_csv_files) ... 

olist_customers_dataset.csv: 99441 rows, 5 columns
olist_geolocation_dataset.csv: 1000163 rows, 5 columns
olist_orders_dataset.csv: 99441 rows, 8 columns
olist_orders_semicolon.csv: 99441 rows, 1 columns
olist_order_items_dataset.csv: 112650 rows, 7 columns
olist_order_payments_dataset.csv: 103886 rows, 5 columns


ok

----------------------------------------------------------------------
Ran 7 tests in 2.533s

OK


olist_order_reviews_dataset.csv: 99224 rows, 7 columns
olist_products_dataset.csv: 32951 rows, 9 columns
olist_sellers_dataset.csv: 3095 rows, 4 columns
product_category_name_translation.csv: 71 rows, 2 columns

========== TEST SUMMARY ==========
Total tests: 7
Passed: 7
Failed: 0
Errors: 0

All unit tests passed successfully!


In [ ]:
# 5, 6

try:
    test_config = load_config()
    print("Configuration is valid")
except ConfigurationError as error:
    logger.error(str(error))
    print("Graceful failure:", error)

Configuration is valid
